# 01 — Prompt Draft: Analista (PRODESP PE 90027/2025)

Itera o prompt do **Agente Analista** manualmente antes de codar `backend/agents/analista.py`.

**Pipeline manual:** PDF → Gemini 2.5 Flash (Extrator) → BigQuery (Qualificador) → Gemini 2.5 Pro (Analista) → `ParecerFinal`.

Refs: `ARCHITECTURE.md` §Agente 3 + §Lógica de Decisão em duas camadas.


In [ ]:
import sys, os, json, textwrap
from pathlib import Path

REPO = Path('/workspaces/lici-adk')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.environ.setdefault('GOOGLE_CLOUD_PROJECT', 'xertica-gen-ai-br')
PROJECT = 'xertica-gen-ai-br'
LOCATION = 'southamerica-east1'

# Drivers disponíveis (escolher um):
EDITAL_PRODESP = REPO / 'PE90027.25-3a-VERSAO-P-A-R-A-P-U-B-L-I-C-A-C-A-O-2.pdf'
EDITAL_CELEPAR = REPO / 'EDITAL RETIFICADO.pdf'

EDITAL_PDF = EDITAL_PRODESP    # troque para EDITAL_CELEPAR para testar o caso strict_match
assert EDITAL_PDF.exists(), f'PDF não encontrado: {EDITAL_PDF}'
print('Driver:', EDITAL_PDF.name, f'({EDITAL_PDF.stat().st_size // 1024} KB)')


## 0. Sanity check — auth + BigQuery

Requer `gcloud auth application-default login` e quota project `xertica-gen-ai-br`.

In [ ]:
from backend.tools.bigquery_tools import sanity_check
sanity_check()


## 1. Extrator — Gemini 2.5 Flash lê o PDF

Input multimodal (PDF nativo). Output: JSON que valida contra `EditalEstruturado`.

In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel, Part, GenerationConfig
from backend.models.schemas import EditalEstruturado

vertexai.init(project=PROJECT, location=LOCATION)

EXTRATOR_SYSTEM_PROMPT = textwrap.dedent('''
Você é o Agente Extrator do lici-adk. Leia o edital anexado (PDF) e devolva APENAS
um JSON válido conforme o schema abaixo — sem prosa antes ou depois.

SCHEMA (campos ausentes = null; listas vazias = []):
{
  "objeto": "descrição objetiva do objeto licitado",
  "orgao": "nome do órgão + UF",
  "uf": "SP",
  "uasg": "533201",
  "modalidade": "Pregão Eletrônico|Adesão a Ata|ETEC|...",
  "data_encerramento": "YYYY-MM-DD",
  "prazo_questionamento": "YYYY-MM-DD",
  "duracao_contrato": "24 meses",
  "valor_estimado": 0.0,
  "portal": "BEC|Comprasnet|Licitações-e|...",
  "requisitos_tecnicos": ["..."],
  "requisitos_habilitacao": ["SICAF", "CND Federal", ...],
  "garantia_contratual": "5% do valor do contrato",
  "nivel_parceria_exigido": "Google Cloud Premier Partner",
  "certificacoes_corporativas_exigidas": ["ISO 27001", ...],
  "certificacoes_profissionais_exigidas": ["Professional Cloud Architect", ...],
  "volumetria_exigida": [{"dimensao": "contas_workspace", "quantidade": 400, "unidade": "usuários"}],
  "modelo_precificacao": ["USN"|"UST"|"USTa"|"bolsa_horas"|"tickets"|"licenca_fixa"|"consumo_volumetria"],
  "tabela_proporcionalidade_ust": {"Especialista": 1.30},
  "nivel_sla_critico": "P1 em 2h úteis, 99,9% disponibilidade",
  "penalidades_glosa_max_pct": 50,
  "exclusividade_me_epp": false,
  "vedacao_consorcio": false,
  "subcontratacao_permitida": "livre|parcial|vedada",
  "exige_poc_mvp": false,
  "prazo_poc": null,
  "modelo_inovacao_etec": false,
  "restricao_temporal_experiencia_meses": 36,
  "localizacao_dados_exigida": "Brasil — BACEN Res. 4.893",
  "dependencias_terceiros_identificadas": ["WhatsApp Business API"],
  "strict_match_atestados": false,
  "match_familia_permitido": true,
  "keywords_busca": ["google cloud", "vertex ai", "chatbot", ...]
}

REGRAS:
- Extraia literalmente do edital. Se um campo não está presente, devolva null ou [].
- `strict_match_atestados=true` SOMENTE se o edital disser explicitamente que não aceita atestados similares.
- `keywords_busca` deve ter 5-10 termos técnicos centrais que o Qualificador usará para buscar atestados.
''').strip()

flash = GenerativeModel('gemini-2.5-flash', system_instruction=EXTRATOR_SYSTEM_PROMPT)

pdf_bytes = EDITAL_PDF.read_bytes()
resp = flash.generate_content(
    [Part.from_data(data=pdf_bytes, mime_type='application/pdf'),
     'Extraia agora o JSON completo do edital.'],
    generation_config=GenerationConfig(temperature=0.1, response_mime_type='application/json'),
)
edital_dict = json.loads(resp.text)
edital = EditalEstruturado.model_validate(edital_dict)

print('Objeto:', (edital.objeto or '')[:200])
print('Órgão :', edital.orgao, '| UF:', edital.uf, '| UASG:', edital.uasg)
print('Modo  :', edital.modalidade, '| duração:', edital.duracao_contrato)
print('Flags :', 'ME/EPP=', edital.exclusividade_me_epp,
      '| consórcio_vedado=', edital.vedacao_consorcio,
      '| strict_match=', edital.strict_match_atestados,
      '| temporal=', edital.restricao_temporal_experiencia_meses)
print('Keywords:', edital.keywords_busca)
edital


## 2. Qualificador — busca BigQuery pelas keywords do Extrator

Roda as 5 queries de `bigquery_tools` para cada keyword, deduplica por id.

In [ ]:
from backend.tools.bigquery_tools import (
    buscar_atestados, buscar_contratos_com_atestado,
    buscar_contratos_sem_atestado, buscar_deals_won, buscar_deals_lost,
    buscar_certificacoes,
)
from backend.models.schemas import QualificadorResult

modo = 'strict' if edital.strict_match_atestados else 'like'
agg = QualificadorResult(modo_busca=modo)

seen_a, seen_c, seen_d, seen_ct = set(), set(), set(), set()

for kw in (edital.keywords_busca or [])[:8]:
    for a in buscar_atestados(kw, mode=modo,
                              restricao_temporal_meses=edital.restricao_temporal_experiencia_meses,
                              limit=20):
        if a.id and a.id not in seen_a:
            agg.atestados.append(a); seen_a.add(a.id)
    for c in buscar_contratos_com_atestado(kw, limit=20):
        key = ('c+a', c.nomedaconta, c.numerodocontrato)
        if key not in seen_c:
            agg.contratos_com_atestado.append(c); seen_c.add(key)
    for c in buscar_contratos_sem_atestado(kw, limit=20):
        key = ('c-a', c.nomedaconta, c.numerodocontrato)
        if key not in seen_c:
            agg.contratos_sem_atestado.append(c); seen_c.add(key)
    for d in buscar_deals_won(kw, limit=10):
        key = ('won', d.conta, d.oportunidade)
        if key not in seen_d:
            agg.deals_won.append(d); seen_d.add(key)
    for d in buscar_deals_lost(kw, limit=5):
        key = ('lost', d.conta, d.oportunidade)
        if key not in seen_d:
            agg.deals_lost.append(d); seen_d.add(key)
    for ct in buscar_certificacoes(kw, limit=30):
        if ct.cert_id and ct.cert_id not in seen_ct:
            agg.certificados.append(ct); seen_ct.add(ct.cert_id)
    agg.queries_executadas += 5

print(f'Modo de busca : {modo}  |  queries: {agg.queries_executadas}')
print(f'Atestados     : {len(agg.atestados)}')
print(f'Contratos c/at: {len(agg.contratos_com_atestado)}  |  Contratos s/at: {len(agg.contratos_sem_atestado)}')
print(f'Deals won     : {len(agg.deals_won)}  |  Deals lost: {len(agg.deals_lost)}')
print(f'Certificados  : {len(agg.certificados)}')


## 3. Analista — Gemini 2.5 Pro com a Lógica de Decisão em 2 camadas

Este prompt vai virar `backend/agents/analista.py`. Itere aqui até o parecer ficar redondo.

In [ ]:
import yaml
PROFILE = yaml.safe_load((REPO / 'backend/xertica_profile.yaml').read_text())

ANALISTA_SYSTEM_PROMPT = textwrap.dedent('''
Você é o Agente Analista do lici-adk — consultor sênior de pré-venda para licitações
públicas da Xertica Brasil. Produz pareceres auditáveis, não marketing.

=================== LÓGICA DE DECISÃO (OBRIGATÓRIA) ===================
Duas camadas SEQUENCIAIS. NUNCA devolva "score 82 mas INAPTO".

CAMADA 1 — Bloqueadores duros (avaliar PRIMEIRO)
Se QUALQUER item abaixo disparar, devolva status=NO-GO ou INAPTO, score_aderencia=null,
preencha `bloqueio_camada_1` com a regra que disparou, e NÃO calcule score:
  1. edital.exclusividade_me_epp=true (Xertica não é ME/EPP) → NO-GO
  2. Edital exige consórcio OBRIGATÓRIO como membro → NO-GO
  3. localizacao_dados_exigida incompatível com `southamerica-*` → INAPTO
  4. Certificação corporativa obrigatória ausente em `certidoes_empresa.certidoes_iso` → INAPTO
  5. Qualificador devolveu ZERO atestados E ZERO contratos para os requisitos técnicos
     centrais E o edital exige atestado como habilitação MANDATÓRIA → INAPTO
  6. nivel_parceria_exigido superior ao de `qualificacao_empresa.parcerias_formais` → INAPTO

CAMADA 2 — Score 0-100 (só roda se Camada 1 passar integralmente)
Soma ponderada (total 100):
  - Cobertura de requisitos técnicos por atestados+contratos (peso 50)
  - Match com `realidade_contratual.objetos_mais_comuns` / contratos âncora (peso 15)
  - Adesão a Ata viável no órgão-alvo (peso 10)
  - Premier Partner ou alinhamento com `especializacoes_google` (peso 10)
  - Certificações profissionais cobrem perfis exigidos (peso 15)
Penalidades:
  - ≥3 deals_lost com mesma Causa_Raiz: -15
  - Cada gap de volumetria numérico: -10
Thresholds: APTO ≥75 | APTO COM RESSALVAS 41-74 | INAPTO ≤40.

=================== REGRA ANTI-ALUCINAÇÃO (#10) ===================
Se o Qualificador devolveu ZERO resultados para um requisito, NUNCA invente capacidade.
Declare GAP TOTAL, score máximo 40, recomende captação de atestado com cliente similar.

=================== EVIDÊNCIAS AUDITÁVEIS ===================
Para cada requisito técnico do edital, preencha `evidencias_por_requisito` com:
  { "requisito": "texto do requisito",
    "fonte_tabela": "atestados"|"contratos"|"closed_deals_won"|"certificados_xertica"|"xertica_profile.yaml",
    "fonte_id": "id do registro ou chave YAML",
    "trecho_literal": "trecho COPIADO do resumodoatestado/resumodocontrato que comprova",
    "tipo_evidencia": "atestado"|"contrato"|"deal_won"|"certificado"|"yaml",
    "confianca": 0.0-1.0 }

=================== OUTPUT (APENAS JSON) ===================
{
  "score_aderencia": null | 0-100,
  "status": "APTO" | "APTO COM RESSALVAS" | "INAPTO" | "NO-GO",
  "bloqueio_camada_1": null | "texto da regra que disparou",
  "requisitos_atendidos": [{"requisito","comprovacao","fonte","link"}],
  "evidencias_por_requisito": [...],
  "gaps": [{"requisito","tipo","delta_numerico","recomendacao"}],
  "estrategia": "recomendação objetiva de participação (2-4 parágrafos)",
  "alertas": ["prazo crítico / risco de glosa / dependência de terceiro / ..."],
  "campos_trello": {"titulo_card": "...", "checklist": ["..."]},
  "edital_orgao": "nome",
  "edital_modalidade": "Pregão..."
}
''').strip()

pro = GenerativeModel('gemini-2.5-pro', system_instruction=ANALISTA_SYSTEM_PROMPT)

payload = {
    'edital': edital.model_dump(),
    'qualificador': agg.model_dump(),
    'xertica_profile': {
        k: PROFILE.get(k) for k in [
            'qualificacao_empresa', 'realidade_contratual', 'volumetria_comprovada',
            'certidoes_empresa', 'especializacoes_google', 'flags_bloqueadoras',
        ] if k in PROFILE
    },
}
payload_json = json.dumps(payload, ensure_ascii=False, default=str)
print(f'Payload: {len(payload_json):,} chars')

resp = pro.generate_content(
    f'ANALISE o edital abaixo. Responda APENAS com o JSON do ParecerFinal.\n\n```json\n{payload_json[:180000]}\n```',
    generation_config=GenerationConfig(temperature=0.2, response_mime_type='application/json'),
)
parecer_raw = resp.text
print('--- Parecer bruto (primeiros 2500 chars) ---')
print(parecer_raw[:2500])


In [ ]:
from backend.models.schemas import ParecerFinal

parecer = ParecerFinal.model_validate_json(parecer_raw)

print('═' * 70)
print(f'Status           : {parecer.status}')
print(f'Score            : {parecer.score_aderencia}')
print(f'Bloqueio Camada 1: {parecer.bloqueio_camada_1}')
print(f'Requisitos OK    : {len(parecer.requisitos_atendidos)}')
print(f'Evidências       : {len(parecer.evidencias_por_requisito)}')
print(f'Gaps             : {len(parecer.gaps)}')
print(f'Alertas          : {len(parecer.alertas)}')
print('═' * 70)
print('\n— ESTRATÉGIA —')
print(parecer.estrategia[:2000])
print('\n— GAPS —')
for g in parecer.gaps:
    print(f'  • [{g.tipo}] {g.requisito}  (Δ={g.delta_numerico})')
    print(f'      → {g.recomendacao}')
print('\n— ALERTAS —')
for a in parecer.alertas:
    print(f'  ⚠ {a}')


## 4. Iteração

1. Ajuste `ANALISTA_SYSTEM_PROMPT` na célula anterior.
2. Re-rode as células 3→parecer.
3. Quando o parecer estabilizar, transcreva o prompt para `backend/agents/analista.py` sem alterar o contrato JSON (`ParecerFinal`).
4. Para testar o caminho de **Camada 1 (bloqueio duro)**, troque `EDITAL_PDF` para `EDITAL_CELEPAR` na célula 1 e re-rode — Celepar tem `strict_match_atestados=true` + restrição temporal 36m + glosa 50% (stress-test real do Analista).